# Day 3 · 실습 2. LSTM Encoder–Decoder로 시퀀스 변환하기

**학습 흐름:** A형에서 Seq2Seq 구조 이해 → B형 1단계에서 상태 전달 따라 하기 → B형 2단계에서 토큰 생성 함수 독립 구현

`A B C → C B A`처럼 입력 시퀀스의 순서를 뒤집는 작은 모델을 사용합니다. **Attention은 사용하지 않습니다.**

- A형은 코드를 순서대로 실행하며 **LSTM의 h·c, Encoder–Decoder 연결, Teacher Forcing, 실제 생성**을 확인합니다.
- B형은 **TODO 부분만 작성**합니다. 결과 확인과 아직 배우지 않은 처리는 완성 코드로 제공합니다.
- 이 노트북과 기존 `weights/reverse_seq2seq.pt` 파일을 실습 폴더에 함께 두세요. Python 3.10 이상과 PyTorch가 필요합니다.

## A형 · LSTM Seq2Seq 구조 이해하기

### A-1. 입력 데이터와 특수 토큰 준비

이번 실습은 A~D 네 가지 토큰을 사용합니다. `<BOS>`는 **Decoder의 시작**, `<EOS>`는 **출력의 끝**을 나타냅니다. `to_ids()`는 토큰을 모델이 읽을 수 있는 정수 ID로 바꿉니다.

In [1]:
from pathlib import Path
import torch
from torch import nn

torch.manual_seed(42)
torch.set_num_threads(1)

# Seq2Seq에서 사용할 단어 목록입니다.
# BOS는 문장의 시작, EOS는 문장의 끝을 나타내는 특수 토큰입니다.
vocab = ["<BOS>", "<EOS>", "A", "B", "C", "D"]

# 각 토큰에 고유한 정수 ID를 부여합니다.
# 예: "<BOS>" → 0, "<EOS>" → 1, "A" → 2
token_to_id = {token: index for index, token in enumerate(vocab)}

# Decoder가 문장 생성의 시작과 종료를 판단할 때 사용할 토큰 ID입니다.
BOS_ID, EOS_ID = token_to_id["<BOS>"], token_to_id["<EOS>"]

# 모델의 주요 설정값입니다.
# VOCAB_SIZE: 예측할 수 있는 전체 토큰 수 (6개)
# EMBEDDING_DIM: 각 토큰을 표현하는 임베딩 벡터의 크기 (16차원)
# HIDDEN_SIZE: LSTM의 Hidden State(h)와 Cell State(c)의 크기 (64차원)
VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE = len(vocab), 16, 64
WEIGHTS = Path("weights/reverse_seq2seq.pt")

# 문자열 토큰을 모델이 처리할 수 있는 정수 ID Tensor로 변환합니다.
def to_ids(tokens):
    # 예: ["A", "B", "C"] → [[2, 3, 4]]
    # 바깥쪽 리스트는 한 번에 처리할 문장의 개수(배치 차원)를 나타냅니다.
    # nn.Embedding의 입력으로 사용하므로 정수형(torch.long)으로 생성합니다.
    return torch.tensor([[token_to_id[token] for token in tokens]], dtype=torch.long)

print("토큰 → ID:", token_to_id)

토큰 → ID: {'<BOS>': 0, '<EOS>': 1, 'A': 2, 'B': 3, 'C': 4, 'D': 5}


### A-2. Encoder와 Decoder의 역할 확인

어제 배운 기본 RNN은 Hidden State(h)를 다음 시점으로 전달했습니다. 반면 **LSTM은 Hidden State(h)와 Cell State(c) 두 상태를 함께 관리**합니다.

이번 실습에서는 LSTM을 이용해 Encoder와 Decoder를 구성하고, 입력 시퀀스의 정보가 출력 시퀀스를 생성하는 데 어떻게 활용되는지 확인합니다.

**아래 코드를 살펴보면서 다음 세 가지에 집중하세요.**

1. **Encoder의 역할:** 입력 시퀀스를 순서대로 읽고, 마지막 Hidden State(h)와 Cell State(c)를 반환합니다.
2. **Decoder의 역할:** Encoder의 마지막 (h, c)를 **초기 상태**로 전달받아 출력 시퀀스를 처리합니다.
3. **Linear 계층의 역할:** Decoder의 각 시점에서 나온 Hidden State를 전체 어휘에 대한 **예측 점수(Logits)**로 변환합니다.

특히 **Encoder가 반환한 (h, c)를 Decoder가 초기 상태로 사용한다는 점**이 중요합니다. 이것이 두 LSTM을 연결하는 기본 Seq2Seq 구조의 핵심입니다.

In [2]:
# Encoder는 입력 문장을 읽고, 마지막 Hidden State(h)와 Cell State(c)를 생성합니다.
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_size=64):
        super().__init__()

        # 입력 토큰 ID를 임베딩 벡터로 변환하는 계층
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 입력 시퀀스를 순서대로 처리하면서 h와 c를 갱신하는 LSTM
        # batch_first=True: 입력을 (배치 크기, 시퀀스 길이, 특징 수) 순서로 처리
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)

    def forward(self, source_ids):
        # 입력 토큰 ID를 임베딩 벡터로 변환합니다.
        embedded = self.embedding(source_ids)
        ######### 핵심 1: LSTM은 시퀀스를 처리한 뒤 마지막 h와 c를 반환합니다.
        # 첫 번째 반환값(각 시점의 출력)은 사용하지 않고, 마지막 상태만 가져옵니다.
        # 이 두 상태는 Decoder에 입력 문장의 정보를 전달하는 역할을 합니다.
        _, (h, c) = self.lstm(embedded)
        return h, c

# Decoder는 Encoder에서 전달받은 상태를 이용해 출력 시퀀스를 예측합니다.
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_size=64):
        super().__init__()

        # Decoder에 입력되는 토큰 ID를 임베딩 벡터로 변환합니다.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Encoder의 h와 c를 초기 상태로 받아 출력 시퀀스를 처리하는 LSTM (지금은 구조만 선언)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        # LSTM의 Hidden State를 전체 어휘에 대한 예측 점수로 변환합니다.
        # 예: 64차원 Hidden State → 6개 토큰의 예측 점수
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, decoder_ids, state):

        # Decoder의 입력 토큰을 임베딩 벡터로 변환합니다.
        embedded = self.embedding(decoder_ids)

        ########### 핵심 2: Encoder에서 전달받은 (h, c)를 LSTM의 초기 상태로 사용합니다.
        # Decoder는 입력 토큰을 처리하면서 h와 c를 계속 갱신합니다.
        # hidden_states: 각 시점의 Hidden State
        # new_state: 마지막 시점에서 갱신된 (h, c) -> 튜플로 묶여 있음. Decoder는 다음 토큰을 처리할 때 이전 시점의 h와 c를 모두 전달받아야 하기 때문.
        hidden_states, new_state = self.lstm(embedded, state)

        ########### 핵심 3: 각 시점의 Hidden State를 토큰별 예측 점수로 변환합니다.
        # 예측 점수가 가장 높은 토큰을 선택하면 다음 단어를 예측할 수 있습니다.
        logits = self.head(hidden_states)
        return logits, new_state

### A-3. Encoder의 상태 전달과 Teacher Forcing

이번에는 Encoder의 마지막 상태를 Decoder에 전달하고, **Teacher Forcing을 이용해 다음 토큰을 예측하는 과정**을 확인합니다.

입력 `A B C`의 정답은 `C B A`이며, Teacher Forcing에서는 이전 예측값 대신 **이전 시점의 실제 정답 토큰**을 Decoder에 입력합니다.

- **Decoder 입력:** `<BOS> C B A`
- **각 시점의 정답:** `C B A <EOS>`

**아래 코드를 살펴보면서 다음 세 가지에 집중하세요.**

1. **Teacher Forcing:** Decoder는 이전 시점의 실제 정답을 입력받아 다음 토큰을 예측합니다.
2. **상태 전달:** Encoder의 마지막 `(h, c)`가 Decoder의 초기 상태로 전달됩니다.
3. **학습 전 예측:** 아직 모델을 학습하지 않았으므로 예측이 틀릴 수 있습니다. 지금은 정확도보다 **데이터 흐름과 상태 전달 과정**을 확인하는 것이 중요합니다.

In [3]:
# 1. Encoder와 Decoder에 사용할 데이터 준비

# Encoder 입력: 원본 시퀀스 A → B → C
source_ids = to_ids(["A", "B", "C"])

# Decoder 입력: BOS로 시작하고, 이전 시점의 정답 토큰을 순서대로 입력
######### 핵심: Teacher Forcing은 모델의 예측값 대신 실제 정답을 다음 입력으로 사용합니다.
decoder_input_ids = to_ids(["<BOS>", "C", "B", "A"])

# Decoder가 각 시점에서 예측해야 하는 정답
# 마지막 EOS는 출력 시퀀스가 끝났음을 의미합니다.
target_ids = to_ids(["C", "B", "A", "<EOS>"])


# 2. Encoder와 Decoder 모델 생성
# 앞에서 정의한 Encoder와 Decoder 클래스를 이용해 각각의 모델을 생성합니다.
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE)


# 3. Encoder → Decoder 상태 전달 및 예측

######## 핵심: Encoder가 입력 시퀀스를 처리하고 마지막 (h, c)를 반환합니다.
state = encoder(source_ids)

######## 핵심: Encoder의 마지막 (h, c)를 Decoder의 초기 상태로 전달합니다.
# Decoder는 Teacher Forcing 입력을 처리하고 각 시점의 예측 점수(Logits)를 계산합니다.
logits, _ = decoder(decoder_input_ids, state)


# 4. 각 시점의 Decoder 입력, 정답, 예측 결과 비교

print("시점 | Decoder 입력 | 정답 | 학습 전 예측")

# 각 시점의 입력 토큰, 정답 토큰, 모델이 예측한 토큰을 비교합니다.
# argmax(-1): 각 시점에서 가장 높은 예측 점수를 가진 토큰 ID를 선택합니다.
for step, (input_id, target_id, pred_id) in enumerate(
    zip(decoder_input_ids[0].tolist(), target_ids[0].tolist(), logits.argmax(-1)[0].tolist()), 1
):
    # 토큰 ID를 실제 문자로 변환해 출력합니다.
    print(f"{step:>4} | {vocab[input_id]:>12} | {vocab[target_id]:>5} | {vocab[pred_id]}")

시점 | Decoder 입력 | 정답 | 학습 전 예측
   1 |        <BOS> |     C | <BOS>
   2 |            C |     B | <BOS>
   3 |            B |     A | <BOS>
   4 |            A | <EOS> | <BOS>


### A-4. 중간 확인: Encoder → Decoder의 Tensor Shape

이번 셀에서만 **입력 ID → Embedding → LSTM → 예측 점수**로 바뀌는 Shape를 확인합니다.

**확인할 핵심:** Encoder의 입력 길이와 Decoder의 입력 길이는 달라도 됩니다. Decoder에 전달되는 것은 Encoder의 *전체 시점 출력*이 아니라 **마지막 h와 c**이며, 이 상태 크기가 Decoder가 요구하는 초기 상태 크기와 맞아야 합니다.

In [4]:
with torch.no_grad():

    # 1. Encoder
    # 입력 토큰 ID를 임베딩 벡터로 변환합니다.
    enc_emb = encoder.embedding(source_ids)

    ###### 핵심: Encoder는 입력 시퀀스 전체를 처리하고 마지막 (h, c)를 반환합니다.
    # enc_outputs: 각 시점에서 나온 Hidden State
    # enc_h, enc_c: 마지막 시점의 Hidden State와 Cell State
    enc_outputs, (enc_h, enc_c) = encoder.lstm(enc_emb)


    # 2. Decoder
    # Decoder에 입력할 토큰을 임베딩 벡터로 변환합니다.
    # 여기서는 Teacher Forcing을 사용하므로 실제 정답 토큰을 입력합니다.
    dec_emb = decoder.embedding(decoder_input_ids)

    ####### 핵심: Encoder의 마지막 (h, c)를 Decoder의 초기 상태로 전달합니다.
    # Decoder는 입력 토큰을 처리하면서 h와 c를 갱신합니다.
    dec_outputs, (dec_h, dec_c) = decoder.lstm(dec_emb, (enc_h, enc_c))

    # 각 시점의 Hidden State를 전체 어휘에 대한 예측 점수로 변환합니다.
    dec_logits = decoder.head(dec_outputs)


    # 3. Encoder의 Tensor Shape 변화 확인

    print("[Encoder]")
    print("입력 ID       :", tuple(source_ids.shape))
    print("Embedding     :", tuple(enc_emb.shape))
    print("LSTM 전체 출력:", tuple(enc_outputs.shape))
    print("마지막 h / c  :", tuple(enc_h.shape), "/", tuple(enc_c.shape))


    # 4. Decoder의 Tensor Shape 변화 확인

    print("\n[Decoder]")
    print("입력 ID       :", tuple(decoder_input_ids.shape))
    print("Embedding     :", tuple(dec_emb.shape))
    print("LSTM 전체 출력:", tuple(dec_outputs.shape))
    print("예측 점수     :", tuple(dec_logits.shape))
    print("갱신된 h / c  :", tuple(dec_h.shape), "/", tuple(dec_c.shape))

[Encoder]
입력 ID       : (1, 3)
Embedding     : (1, 3, 16)
LSTM 전체 출력: (1, 3, 64)
마지막 h / c  : (1, 1, 64) / (1, 1, 64)

[Decoder]
입력 ID       : (1, 4)
Embedding     : (1, 4, 16)
LSTM 전체 출력: (1, 4, 64)
예측 점수     : (1, 4, 6)
갱신된 h / c  : (1, 1, 64) / (1, 1, 64)


### A-5. Encoder와 Decoder는 함께 학습합니다

Decoder의 예측에서 계산한 오차는 Encoder가 전달한 `(h, c)`를 통해 **Encoder까지 역전파**됩니다. 따라서 두 모델의 가중치를 함께 학습할 수 있습니다.

**아래 코드를 살펴보면서 다음 세 가지에 집중하세요.**

1. **공동 학습:** Encoder와 Decoder의 파라미터를 하나의 Optimizer에 등록합니다.
2. **역전파:** Decoder의 손실이 `(h, c)`를 거쳐 Encoder까지 전달됩니다.
3. **기울기 확인:** 두 모델 모두 기울기가 계산되어 학습 신호가 전달되었는지 확인합니다.

이번 실습에서는 **한 번의 가중치 갱신**으로 두 모델의 학습 연결을 확인합니다.

In [5]:
# 1. 학습 준비

# Encoder와 Decoder의 모든 파라미터를 하나의 Optimizer에 등록합니다.
####### 핵심: 두 모델을 함께 학습하기 위해 두 모델의 가중치를 모두 포함합니다.
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=0.005
)

# 각 시점에서 예측한 토큰과 실제 정답 사이의 오차를 계산하는 손실 함수
loss_fn = nn.CrossEntropyLoss()

# Encoder와 Decoder를 학습 모드로 전환합니다.
encoder.train()
decoder.train()


# 2. 순전파 및 손실 계산

# 이전 학습 단계에서 계산된 기울기를 초기화합니다.
optimizer.zero_grad()

####### 핵심 : Encoder의 마지막 (h, c)를 Decoder의 초기 상태로 전달합니다.
# Decoder는 각 시점의 토큰에 대한 예측 점수(Logits)를 계산합니다.
logits, _ = decoder(decoder_input_ids, encoder(source_ids))

# Decoder의 예측 점수와 실제 정답을 비교하여 손실을 계산합니다.
# 모든 시점의 예측을 한꺼번에 비교하기 위해 Tensor 형태를 변환합니다.
# logits: (1, 4, 6) → (4, 6) / target_ids: (1, 4) → (4,)
loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))


# 3. 역전파 및 기울기 확인

####### 핵심 : Decoder에서 계산한 손실을 역전파합니다.
# Encoder의 (h, c)를 통해 Decoder와 연결되어 있으므로
# 학습 신호가 Decoder뿐 아니라 Encoder까지 전달됩니다.
loss.backward()

# Encoder의 LSTM 입력 가중치에 기울기가 계산되었는지 확인합니다.
encoder_grad = encoder.lstm.weight_ih_l0.grad.abs().sum().item()

# Decoder의 LSTM 입력 가중치에도 기울기가 계산되었는지 확인합니다.
decoder_grad = decoder.lstm.weight_ih_l0.grad.abs().sum().item()


# 4. 가중치 갱신

####### 핵심 : 계산된 기울기를 이용해 Encoder와 Decoder의 가중치를 함께 갱신합니다.
optimizer.step()


# 5. 학습 결과 확인

# 현재 학습 단계에서 계산한 손실값을 출력합니다.
print(f"학습 손실: {loss.item():.4f}")

# 기울기 절댓값의 합이 0보다 크면 해당 가중치에 학습 신호가 전달된 것입니다.
print("Encoder까지 기울기 전달:", encoder_grad > 0)
print("Decoder에도 기울기 전달:", decoder_grad > 0)

학습 손실: 1.7965
Encoder까지 기울기 전달: True
Decoder에도 기울기 전달: True


### A-6. 미리 학습된 모델로 교체하기

앞에서는 **상태 연결과 역전파의 원리**만 확인했습니다. 이제 역순 변환을 학습해 둔 가중치를 불러와, 실제로 토큰이 생성되는 과정을 살펴봅니다.

In [6]:
# 기존 실습에서 제공된 weights/reverse_seq2seq.pt를 사용합니다.
checkpoint = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])
encoder.eval()
decoder.eval()
print("학습된 Encoder·Decoder 가중치 불러오기 완료")

학습된 Encoder·Decoder 가중치 불러오기 완료


### A-7. Decoder 실제로 생성하기

학습할 때와 달리, **생성할 때는 다음 정답을 알 수 없습니다.** Decoder는 `<BOS>`로 시작하고, **방금 예측한 토큰을 다음 입력으로 다시 사용**합니다. 이때 `(h, c)`도 매 시점 갱신됩니다.

A-3의 *정답을 다음 입력으로 사용한 방식*과 아래 코드를 비교해 보세요.

In [7]:
# 학습이 아닌 실제 예측 과정이므로 기울기를 계산하지 않습니다.
with torch.no_grad():

    # 1. Encoder의 마지막 (h, c)를 Decoder의 초기 상태로 준비합니다.
    state = encoder(source_ids)

    # Decoder는 문장의 시작을 나타내는 BOS 토큰부터 입력받습니다.
    current_token = torch.tensor([[BOS_ID]], dtype=torch.long)

    print("시점 | Decoder 입력 | 예측 토큰")

    # 2. Decoder가 토큰을 하나씩 예측하는 과정을 반복합니다.
    # 최대 10번까지 생성하며, EOS가 나오면 반복을 종료합니다.
    for step in range(1, 11):

        ######## 핵심: 현재 토큰과 (h, c)를 Decoder에 입력합니다.
        # Decoder가 다음 토큰을 예측하고, 갱신된 (h, c)를 state에 저장합니다.
        logits, state = decoder(current_token, state)

        ####### 핵심 : 마지막 시점의 예측 점수 중 가장 높은 토큰을 선택합니다.
        # argmax는 가장 높은 점수를 가진 토큰의 ID를 반환합니다.
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

        # 현재 입력 토큰과 모델이 예측한 다음 토큰을 비교합니다.
        print(f"{step:>4} | {vocab[current_token.item()]:>12} | {vocab[next_token.item()]}")

        # 3.  EOS가 예측되면 문장 생성이 완료된 것이므로 반복을 종료합니다.
        if next_token.item() == EOS_ID:
            break

        # ####### 핵심 : Teacher Forcing과 달리 모델이 예측한 토큰을 다음 입력으로 사용합니다.
        # 이 과정을 반복하면서 Decoder가 출력 시퀀스를 한 토큰씩 생성합니다.
        current_token = next_token

시점 | Decoder 입력 | 예측 토큰
   1 |        <BOS> | C
   2 |            C | B
   3 |            B | A
   4 |            A | <EOS>


## B형 1단계 · Encoder–Decoder 연결 따라 하기

**문제:** 입력 `A B C D`를 Encoder에 넣고, 마지막 `(h, c)`를 Decoder의 초기 상태로 전달하여 예측 점수(`logits`)를 반환하세요.

**A-3의 상태 전달 코드 두 줄을 그대로 가져오면 됩니다.** Decoder 입력과 정답 토큰은 미리 준비했습니다.

In [10]:
# 문제에 사용할 입력과 정답은 제공됩니다.
source_b = to_ids(["A", "B", "C", "D"])
decoder_input_b = to_ids(["<BOS>", "D", "C", "B", "A"])
target_b = to_ids(["D", "C", "B", "A", "<EOS>"])

def teacher_forward(encoder, decoder, source_ids, decoder_input_ids):
    state = encoder(source_ids)
    logits, _ = decoder(decoder_input_ids,state)

    return logits
    

In [11]:
# 결과 확인: B형 1단계 함수를 완성한 뒤 실행하세요.
with torch.no_grad():
    logits_b = teacher_forward(encoder, decoder, source_b, decoder_input_b)

print("Decoder 입력:", [vocab[i] for i in decoder_input_b[0].tolist()])
print("정답        :", [vocab[i] for i in target_b[0].tolist()])
print("모델 예측   :", [vocab[i] for i in logits_b.argmax(-1)[0].tolist()])
assert logits_b.shape == (1, 5, VOCAB_SIZE), "Encoder–Decoder 연결을 확인하세요."
print("B형 1단계: 실행 완료")

Decoder 입력: ['<BOS>', 'D', 'C', 'B', 'A']
정답        : ['D', 'C', 'B', 'A', '<EOS>']
모델 예측   : ['D', 'C', 'B', 'A', '<EOS>']
B형 1단계: 실행 완료


## B형 2단계 · 토큰 생성 함수 독립 구현

**문제:** A-7에서 확인한 **BOS 시작 → Decoder 실행 → 다음 토큰 선택 → 예측 토큰 재입력** 과정을 조합해 `generate_tokens()`를 완성하세요.

A형에서 배운 **핵심 네 줄만 직접 구현**합니다. EOS 종료 처리, 결과 저장, 반복 횟수 제한은 이미 제공되어 있습니다. 함수는 BOS/EOS를 제외한 **토큰 ID 목록**을 반환합니다.

In [14]:
@torch.no_grad()
def generate_tokens(decoder, initial_state, bos_id, eos_id, max_new_tokens=10):
    
    state = initial_state
    generated_ids = []

    current_token = torch.tensor([[bos_id]],dtype=torch.long)

    for _ in range(max_new_tokens):
        
        logits, state = decoder(current_token,state)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

        # EOS를 만나면 끝내고, 그 밖의 토큰만 결과에 저장합니다.
        if next_token.item() == eos_id:
            break
        generated_ids.append(int(next_token.item()))

        current_token = next_token
        
    return generated_ids

In [15]:
# 결과 확인: B형 2단계 함수를 완성한 뒤 실행하세요.
examples = [
    ("A B C", "C B A"),
    ("A B C D", "D C B A"),
]

print("입력       | 생성 결과  | 확인")
for input_text, expected in examples:
    with torch.no_grad():
        initial_state = encoder(to_ids(input_text.split()))
    predicted_ids = generate_tokens(decoder, initial_state, BOS_ID, EOS_ID)
    predicted = " ".join(vocab[i] for i in predicted_ids)
    print(f"{input_text:<10} | {predicted:<10} | {'일치' if predicted == expected else '확인 필요'}")
    assert predicted == expected, "예측 토큰 재입력과 상태 갱신을 확인하세요."

print("B형 2단계: 실행 완료")

입력       | 생성 결과  | 확인
A B C      | C B A      | 일치
A B C D    | D C B A    | 일치
B형 2단계: 실행 완료


### 실습 마무리

이번 실습에서는 **LSTM을 이용한 Seq2Seq의 전체 구조와 학습·생성 과정**을 코드로 확인했습니다.

**이번 실습에서 반드시 기억해야 할 핵심 내용은 다음과 같습니다.**

1. **LSTM의 상태 관리:** LSTM은 Hidden State(h)와 Cell State(c)를 함께 관리하며, 입력 시퀀스의 정보를 다음 시점으로 전달합니다.
2. **Encoder–Decoder 연결:** Encoder는 입력 시퀀스를 처리하고, 마지막 `(h, c)`를 Decoder의 초기 상태로 전달합니다. Decoder는 이를 이용해 출력 시퀀스를 생성합니다.
3. **Encoder와 Decoder의 공동 학습:** Decoder에서 계산한 손실은 `(h, c)`를 통해 Encoder까지 역전파되어 두 모델이 함께 학습됩니다.
4. **Teacher Forcing과 실제 생성:** 학습할 때는 이전 시점의 정답을 입력하지만, 실제 생성할 때는 **모델이 예측한 토큰을 다음 입력으로 사용**하고 EOS가 나올 때까지 반복합니다.

**결국 Seq2Seq의 핵심은 Encoder가 입력 시퀀스의 정보를 전달하고, Decoder가 그 정보를 바탕으로 새로운 출력 시퀀스를 만들어 낸다는 것입니다.**